This notebook performs cell phenotyping analysis, similar to the NSCLC Cancer Cell paper

In [1]:
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import pickle as pkl
import skimage
import yaml
from typing import Union, Optional, Type, Tuple, List, Dict
import sys
from skimage.color import label2rgb
import json
# import nrrd

import pandas as pd
import seaborn as sns
# Project Root
# used for searching packages and functions
# TODO: enter your project root dir here
ROOT_DIR = '/project/Xie_Lab/zgu/xiao_multiplex/multiTAP/image_cytof'

sys.path.append(ROOT_DIR)
sys.path.append(os.path.join(ROOT_DIR, 'image_cytof'))
from cytof.hyperion_preprocess import cytof_read_data_roi
from cytof.utils import save_multi_channel_img, check_feature_distribution
from cytof.classes import CytofImageTiff
from cytof.classes import CytofCohort


/project/Xie_Lab/zgu/conda_stuff/envs/cytof-shared/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SAVED_GROUPS = [86, 87, 88, 175, 176, 178]
BASE_CSV_DIR = "/project/Xie_Lab/zgu/xiao_multiplex/nsclc_multiTAP_work"
accuml_type = 'sum'
cell_suffix = f"cell_{accuml_type}"

# pt id and ROI id mapping
roi_pt_id_mapping = pd.read_csv('/project/Xie_Lab/zgu/xiao_multiplex/nsclc_multiTAP_work/roi_pt_id_mapping.csv')


In [3]:
bin_df_list = list()

# merge the binary dfs
for prefix in SAVED_GROUPS:
    csv_path = os.path.join(BASE_CSV_DIR, f"nsclc_save_group{prefix}", f"nsclc_save_group{prefix}_binary_expr_df_{accuml_type}.csv")
    df = pd.read_csv(csv_path)
    bin_df_list.append(df)

combined_binary_df = pd.concat(bin_df_list, axis=0, ignore_index=True)
# combined_spatial_df['roi_mappable'] = combined_spatial_df['ROI_ID'].apply(extract_roi_id)
combined_binary_df

,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,anti-Hu_1950((2832))Yb173-Yb173_cell_sum,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
1,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
3,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3950215,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
3950216,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
3950217,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
3950218,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15


In [4]:
# len(np.unique(combined_binary_df['pt_id']))
tumor_marker = f"panCyto_234((2745))Lu175-Lu175_{cell_suffix}"
# first find the tumor cells by panCK
tumor_cells = combined_binary_df[combined_binary_df[tumor_marker]].reset_index(drop=True)
non_tumor_cells = combined_binary_df[~combined_binary_df[tumor_marker]].reset_index(drop=True)
print(len(tumor_cells), "tumors cells identified")



1040149 tumors cells identified


In [5]:
# remove cells with both CD3 and CD20 positive
cd3_marker = f"CD3_1841((3363))Sm152-Sm152_{cell_suffix}"
cd20_marker = f"CD20_36((3369))Sm149-Sm149_{cell_suffix}"
non_tumor_cleaned = non_tumor_cells[~(non_tumor_cells[cd3_marker] & non_tumor_cells[cd20_marker])]
non_tumor_cleaned

,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,anti-Hu_1950((2832))Yb173-Yb173_cell_sum,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
4,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2910066,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
2910067,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
2910068,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15
2910069,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15


In [6]:
print(len(non_tumor_cleaned[non_tumor_cleaned[cd3_marker]]))
print(len(non_tumor_cleaned[non_tumor_cleaned[cd20_marker]]))

71774
76040


In [20]:
non_tumor_cleaned["final_cell_type"] = np.nan

# assign CD4 Treg (CD3+/CD4+/FOXP3+)
cd4_marker = f"CD4_2293((3000))Yb171-Yb171_{cell_suffix}"
foxp3_marker = f"FOXP3_115((2911))Dy163-Dy163_{cell_suffix}"

cd4_t_reg_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd4_marker] & non_tumor_cleaned[foxp3_marker]
non_tumor_cleaned.loc[cd4_t_reg_index, "final_cell_type"] = "CD4 Treg"
print(np.sum(cd4_t_reg_index), "CD4 T reg identified")

# assign IDO positive subsets
ido_marker = f"Indolea_2281((3014))Eu151-Eu151_{cell_suffix}"
ido_cd4_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd4_marker] & non_tumor_cleaned[ido_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[ido_cd4_index, "final_cell_type"] = "IDO_CD4"
print(np.sum(ido_cd4_index), "IDO_CD4 identified")

cd8_marker = f"CD8a_1718((2991))Er166-Er166_{cell_suffix}"
ido_cd8_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd8_marker] & non_tumor_cleaned[ido_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[ido_cd8_index, "final_cell_type"] = "IDO_CD8"
print(np.sum(ido_cd8_index), "IDO_CD8 identified")

# assign proliferation (Ki-67) subsets
ki67_marker = f"Ki-67_142((3418))Pt194-Pt194_{cell_suffix}"
ki67_cd4_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd4_marker] & non_tumor_cleaned[ki67_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[ki67_cd4_index, "final_cell_type"] = "Ki67_CD4"
print(np.sum(ki67_cd4_index), "Ki67 CD4 identified")

ki67_cd8_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd8_marker] & non_tumor_cleaned[ki67_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[ki67_cd8_index, "final_cell_type"] = "Ki67_CD8"
print(np.sum(ki67_cd8_index), "Ki67 CD8 identified")

# assign TCF1/7 subsets
# TCF7 gene also known as TCF-1 
tcf7_marker = f"TCF1TCF_2221((3415))Gd160-Gd160_{cell_suffix}"
tcf7_cd4_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd4_marker] & non_tumor_cleaned[tcf7_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[tcf7_cd4_index, "final_cell_type"] = "TCF1/7 CD4"
print(np.sum(tcf7_cd4_index), "TCF1/7 CD4 identified")

tcf7_cd8_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd8_marker] & non_tumor_cleaned[tcf7_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[tcf7_cd8_index, "final_cell_type"] = "TCF1/7 CD8"
print(np.sum(tcf7_cd8_index), "TCF1/7 CD8 identified")

# assign PD-1 subsets
pd1_marker = f"CD279(P_1743((3414))Gd155-Gd155_{cell_suffix}"
pd1_cd4_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd8_marker] & non_tumor_cleaned[pd1_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[pd1_cd4_index, "final_cell_type"] = "PD1 CD4"
print(np.sum(pd1_cd4_index), "PD1 CD4 identified")

# assign general CD4 and CD8
cd4_only_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd4_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[cd4_only_index, "final_cell_type"] = "CD4"
print(np.sum(cd4_only_index), "CD4 general identified")

cd8_only_index = non_tumor_cleaned[cd3_marker] & non_tumor_cleaned[cd8_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[cd8_only_index, "final_cell_type"] = "CD8"
print(np.sum(cd8_only_index), "CD8 general identified")

non_tumor_cleaned

/tmp/ipykernel_114504/3436617015.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_tumor_cleaned["final_cell_type"] = np.nan


2256 CD4 T reg identified
305 IDO_CD4 identified
1598 IDO_CD8 identified
572 Ki67 CD4 identified
2562 Ki67 CD8 identified
89 TCF1/7 CD4 identified
681 TCF1/7 CD8 identified
220 PD1 CD4 identified
907 CD4 general identified
25486 CD8 general identified


,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id,final_cell_type
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
4,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2910066,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910067,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910068,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910069,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN


In [21]:
# above assignments for T cells only
# allow assignment only in np.nan, so by defn it excludes T cells
hla_marker = f"HLA-DR_1849((3362))Nd143-Nd143_{cell_suffix}"
cd68_marker = f"CD68_77((3413))Nd150-Nd150_{cell_suffix}"
myeloid_index = non_tumor_cleaned[hla_marker] & non_tumor_cleaned[cd68_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[myeloid_index, "final_cell_type"] = "myeloid"
print(np.sum(myeloid_index), "myeloid cells identified")

# assign neutrophils
mpo_marker = f"Myelope_276((2996))Y89-Y89_{cell_suffix}"
mmp9_marker = f"MMP9_2241((2912))Gd158-Gd158_{cell_suffix}"
neutrophils_index = non_tumor_cleaned[mpo_marker] & non_tumor_cleaned[mmp9_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[neutrophils_index, "final_cell_type"] = "neutrophils"
print(np.sum(neutrophils_index), "neutrophils identified")

# assign B cells
b_cell_index = non_tumor_cleaned[cd20_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[b_cell_index, "final_cell_type"] = "B cells"
print(np.sum(b_cell_index), "B cells identified")

non_tumor_cleaned

60147 myeloid cells identified
31012 neutrophils identified
72278 B cells identified


,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id,final_cell_type
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
4,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2910066,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910067,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910068,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910069,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN


In [22]:
# subset vessel cells
# first find lymphatic endothelial
lyve_marker = f"LYVE-1_1982((2881))Er168-Er168_{cell_suffix}"
ccl21_marker = f"CCL21 6_2177((2889))Yb174-Yb174_{cell_suffix}"
lymphatic_index = non_tumor_cleaned[lyve_marker] & non_tumor_cleaned[ccl21_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[lymphatic_index, "final_cell_type"] = "lymphatic endothelial"
print(np.sum(lymphatic_index), "lymphatic cells identified")

# then overwrite lymphatic with HEV if PNAd+
pnad_marker = f"PNAd_1981((3323))Ho165-Ho165_{cell_suffix}"
hev_cells_index = non_tumor_cleaned[pnad_marker] & lymphatic_index
non_tumor_cleaned.loc[hev_cells_index, "final_cell_type"] = "high endothelial venules (HEV)"
print(np.sum(hev_cells_index), "HEVs identified")

# get general endothelial cells (CD146+/CD31+/vWF+/CCL21-)
cd146_marker = f"CD146_22((3259))Nd144-Nd144_{cell_suffix}"
cd31_vwf_marker = f"CD31_1859((3370))Yb172-Yb172_{cell_suffix}"
endothelial_index = non_tumor_cleaned[cd146_marker] & non_tumor_cleaned[cd31_vwf_marker] & (~non_tumor_cleaned[ccl21_marker]) & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[endothelial_index, "final_cell_type"] = "endothelial"
print(np.sum(endothelial_index), "endothelial cells identified")

non_tumor_cleaned

3135 lymphatic cells identified
186 HEVs identified
35700 endothelial cells identified


,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id,final_cell_type
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
4,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2910066,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910067,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910068,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910069,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN


In [27]:
# check cell assignments so far
temp = non_tumor_cleaned[~non_tumor_cleaned['final_cell_type'].isna()]
print(np.unique(temp['final_cell_type'], return_counts=True))

# CAF assignments
# mCAF
fap_marker = f"fap_323((3412))Nd142-Nd142_{cell_suffix}"
mmp11_marker = f"MMP11_2925((3364))Sm154-Sm154_{cell_suffix}"
collagen_marker = f"Collage_1360((2568))Sm147-Sm147_{cell_suffix}"
mcaf_index = non_tumor_cleaned[fap_marker] & non_tumor_cleaned[mmp11_marker] & non_tumor_cleaned[collagen_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[mcaf_index, "final_cell_type"] = "mCAF"
print(np.sum(mcaf_index), "mCAF identified")

# iCAF
cd34_marker = f"CD34_2254((3337))Er170-Er170_{cell_suffix}"
cd248_marker = f"CD248 E_2178((2830))Er167-Er167_{cell_suffix}"
icaf_index = non_tumor_cleaned[cd34_marker] & non_tumor_cleaned[cd248_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[icaf_index, "final_cell_type"] = "iCAF"
print(np.sum(icaf_index), "iCAF identified")

# tCAF
cd10_marker = f"CD10_2546((3029))Dy161-Dy161_{cell_suffix}"
cd73_marker = f"CD73_2193((3319))Gd156-Gd156_{cell_suffix}"
tcaf_index = non_tumor_cleaned[cd10_marker] & non_tumor_cleaned[cd73_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[tcaf_index, "final_cell_type"] = "tCAF"
print(np.sum(tcaf_index), "tCAF identified")

# hypoxic CAF
caix_marker = f"Carboni_2443((2757))Nd146-Nd146_{cell_suffix}"
hypoxic_caf_index = non_tumor_cells[caix_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[hypoxic_caf_index, "final_cell_type"] = "hypoxic CAF"
print(np.sum(hypoxic_caf_index), "hypoxic CAF identified")

# hypoxic tCAF
# reassign classified hypoxic CAF to hypoxic tCAF if CD10+
hypoxic_tcaf_index = non_tumor_cells[cd10_marker] & hypoxic_caf_index
non_tumor_cleaned.loc[hypoxic_tcaf_index, "final_cell_type"] = "hypoxic tCAF"
print(np.sum(hypoxic_tcaf_index), "hypoxic tCAF identified")

#ifnCAF
ifn_caf_index = non_tumor_cells[ido_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[ifn_caf_index, "final_cell_type"] = "IFN CAF"
print(np.sum(ifn_caf_index), "IFN CAF identified")

# vCAF
vcaf_index = non_tumor_cells[cd146_marker] & (~non_tumor_cells[cd34_marker]) & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[vcaf_index, "final_cell_type"] = "vCAF"
print(np.sum(vcaf_index), "vCAF identified")

# dCAF
dcaf_index = non_tumor_cells[ki67_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[dcaf_index, "final_cell_type"] = "dCAF"
print(np.sum(dcaf_index), "dCAF identified")

# PDPN CAF
pdpn_marker = f"Podopla_1463((2619))Eu153-Eu153_{cell_suffix}"
pdpn_caf_index = non_tumor_cleaned[pdpn_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[pdpn_caf_index, "final_cell_type"] = "PDPN CAF"
print(np.sum(pdpn_caf_index), "PDPN CAF identified")

# SMA CAF
sma_marker = f"SMA_174((3277))In115-In115_{cell_suffix}"
sma_caf_index = non_tumor_cleaned[sma_marker] & (non_tumor_cleaned['final_cell_type'].isna())
non_tumor_cleaned.loc[sma_caf_index, "final_cell_type"] = "SMA CAF"
print(np.sum(sma_caf_index), "SMA CAF identified")

non_tumor_cleaned

(array(['B cells', 'CD4', 'CD4 Treg', 'CD8', 'IDO_CD4', 'IDO_CD8',
       'Ki67_CD4', 'Ki67_CD8', 'PD1 CD4', 'TCF1/7 CD4', 'TCF1/7 CD8',
       'endothelial', 'high endothelial venules (HEV)',
       'lymphatic endothelial', 'myeloid', 'neutrophils'], dtype=object), array([72278,   907,  2256, 25486,   305,  1598,   572,  2562,   220,
          89,   681, 35700,   186,  2949, 60147, 31012]))
11235 mCAF identified
11360 iCAF identified
8133 tCAF identified
70345 hypoxic CAF identified
7313 hypoxic tCAF identified
32065 IFN CAF identified
32365 vCAF identified
62838 dCAF identified
29755 PDPN CAF identified
281459 SMA CAF identified


,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id,final_cell_type
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,NaN
4,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,86_1,NSCLC_ALL_86_A_A1_1,SMA CAF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2910066,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910067,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910068,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN
2910069,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,178_571,NSCLC_ALL_178_C_C_4_15,NaN


In [26]:
ido_marker

'Indolea_2281((3014))Eu151-Eu151_cell_sum'

In [41]:
temp = non_tumor_cleaned[non_tumor_cleaned[cd4_marker] & non_tumor_cleaned[cd3_marker]]

temp = temp[[cd3_marker, foxp3_marker, "final_cell_type"]]

temp['final_cell_type'].isna()

3558        True
4723        True
4727       False
4730       False
4742       False
           ...  
2907926    False
2907944    False
2908107     True
2908189    False
2909204     True
Name: final_cell_type, Length: 4149, dtype: bool

In [42]:
non_tumor_cleaned['final_cell_type'].isna()

0          True
1          True
2          True
3          True
4          True
           ... 
2910066    True
2910067    True
2910068    True
2910069    True
2910070    True
Name: final_cell_type, Length: 2895173, dtype: bool

In [20]:
len(cd3_cd20_pos) / len(non_tumor_cells)
len(non_tumor_cells)

2910071

# scratch

In [45]:
all_tumor_cells = 0
all_nontumor_cells = 0
accumul_type = 'sum'

for prefix in SAVED_GROUPS:
    prefix_pt_roi_path = f'/project/Xie_Lab/zgu/xiao_multiplex/nsclc_multiTAP_work/nsclc_save_group{prefix}/nsclc_save_group{prefix}.pkl'
    cytof_cohort_whole_slide = pkl.load(open(prefix_pt_roi_path, 'rb'))
    pt_prefix_rois = roi_pt_id_mapping[roi_pt_id_mapping['ROI'].str.startswith(f'{prefix}_')].reset_index(drop=True)
    prefix_pt_ids = np.unique([pt_prefix_rois['Patient_ID']])
    print(f"\n{len(prefix_pt_ids)} unique patient IDs identified in 'nsclc_save_group{prefix}.pkl' file")

    prefix_pt_ids = prefix_pt_ids[:5]

    save_group_df_list = list()
    # process for each patient
    for pt_id in prefix_pt_ids:
        print('\nProcessing Patient ID', pt_id)
        per_pt_roi_dict = dict() # to be pass into CytofCohort later

        # load the pt's ROIs
        df_to_load = pt_prefix_rois[pt_prefix_rois['Patient_ID']==pt_id]
        print(len(df_to_load), 'ROIs identified for patient', pt_id)

        try:
            # load all of this pt's ROI into a new dict
            for index, row in df_to_load.iterrows():
                new_key = f"{row['SLIDE']}_{row['ROI']}"
                per_pt_roi_dict[new_key] = cytof_cohort_whole_slide.cytof_images[new_key]

            # df_cohort not saved, creating one automatically from CytofCohort
            per_pt_cohort = CytofCohort(cytof_images=per_pt_roi_dict, dir_out=None)
            per_pt_cohort.batch_process_feature()
            per_pt_cohort.generate_summary(accumul_type=accumul_type)
                
            # go through each roi, get their binary marker-cell expression
            for key, cytof_img in per_pt_cohort.cytof_images.items():
                
                # get the mean expression
                pt_binary_df = cytof_img.get_binary_pos_express_df(feature_name='75normed', accumul_type=accumul_type)
                
                save_binary_df = pt_binary_df.copy()
                save_binary_df['pt_id'] = pt_id
                save_binary_df['roi_id'] = key

                save_group_df_list.append(save_binary_df)
                
                panck_col = "panCyto_234((2745))Lu175-Lu175_cell_sum"

                # mean panCK expression used to identify tumor cells
                # then count number of cells with positively expressed panCK
                per_pt_tumor_cells = np.sum(pt_binary_df[panck_col])
                per_pt_nontumor_cells = len(pt_binary_df) - per_pt_tumor_cells

                # tally up
                all_tumor_cells += per_pt_tumor_cells
                all_nontumor_cells += per_pt_nontumor_cells


        except Exception as e:
            print(f'pt_id {pt_id} not processed due to error {e}')

    
    print("# of tumor cells so far:", all_tumor_cells)
    print("# of nontumor cells so far:", all_nontumor_cells)

    # concatenate to one df at the group level
    save_group_binary_df = pd.concat(save_group_df_list)

    # save to file
    save_path = os.path.join(BASE_PKL_DIR, f'nsclc_save_group{prefix}', f'XXXnsclc_save_group{prefix}_binary_expr_df_{accumul_type}.csv')
    save_group_binary_df.to_csv(save_path, index=False)
    
    


139 unique patient IDs identified in 'nsclc_save_group86.pkl' file

Processing Patient ID 86_1
2 ROIs identified for patient 86_1
Getting thresholds for cell sum of all markers.

Processing Patient ID 86_10
1 ROIs identified for patient 86_10
Getting thresholds for cell sum of all markers.

Processing Patient ID 86_100
2 ROIs identified for patient 86_100
Getting thresholds for cell sum of all markers.

Processing Patient ID 86_101
2 ROIs identified for patient 86_101
Getting thresholds for cell sum of all markers.

Processing Patient ID 86_102
2 ROIs identified for patient 86_102
Getting thresholds for cell sum of all markers.
# of tumor cells so far: 4963
# of nontumor cells so far: 10756

192 unique patient IDs identified in 'nsclc_save_group87.pkl' file

Processing Patient ID 87_162
2 ROIs identified for patient 87_162
Getting thresholds for cell sum of all markers.

Processing Patient ID 87_163
1 ROIs identified for patient 87_163
Getting thresholds for cell sum of all markers.



In [48]:
from datetime import datetime
current_datetime = datetime.now()
formatted_datetime = current_datetime.strftime("%Y-%m-%d %H:%M:%S")
formatted_datetime

print(formatted_datetime, f"file saved to {save_path}")

2025-08-20 17:07:03 file saved to /project/Xie_Lab/zgu/xiao_multiplex/nsclc_multiTAP_work/nsclc_save_group87/XXXnsclc_save_group87_binary_expr_df_sum.csv


In [41]:
concatenated_df = pd.concat(save_group_df_list)
concatenated_df

,Myelope_276((2996))Y89-Y89_cell_sum,FSP1 S1_2263((3411))In113-In113_cell_sum,SMA_174((3277))In115-In115_cell_sum,117Sn-Sn117_cell_sum,Histone_126((2797))Pr141-Pr141_cell_sum,fap_323((3412))Nd142-Nd142_cell_sum,HLA-DR_1849((3362))Nd143-Nd143_cell_sum,CD146_22((3259))Nd144-Nd144_cell_sum,Cadheri_2088((2893))Nd145-Nd145_cell_sum,Carboni_2443((2757))Nd146-Nd146_cell_sum,...,anti-Hu_1950((2832))Yb173-Yb173_cell_sum,CCL21 6_2177((2889))Yb174-Yb174_cell_sum,panCyto_234((2745))Lu175-Lu175_cell_sum,K-Cadhe_2600((3417))Yb176-Yb176_cell_sum,Ki-67_142((3418))Pt194-Pt194_cell_sum,Caveoli_1945((2899))Pt195-Pt195_cell_sum,206Pb-Pb206_cell_sum,CD15_627((2997))Bi209-Bi209_cell_sum,pt_id,roi_id
0,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,87_162,NSCLC_ALL_87_A_A1_1
1,False,False,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,87_162,NSCLC_ALL_87_A_A1_1
2,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,87_162,NSCLC_ALL_87_A_A1_1
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,87_162,NSCLC_ALL_87_A_A1_1
4,False,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,87_162,NSCLC_ALL_87_A_A1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2979,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,87_167,NSCLC_ALL_87_A_A4_2
2980,False,False,False,True,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,87_167,NSCLC_ALL_87_A_A4_2
2981,False,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,87_167,NSCLC_ALL_87_A_A4_2
2982,False,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,True,87_167,NSCLC_ALL_87_A_A4_2


In [43]:
# np.unique(concatenated_df['slide_roi'], return_counts=True)
np.unique(concatenated_df['roi_id'], return_counts=True)

(array(['NSCLC_ALL_87_A_A1_1', 'NSCLC_ALL_87_A_A1_2',
        'NSCLC_ALL_87_A_A2_1', 'NSCLC_ALL_87_A_A2_2',
        'NSCLC_ALL_87_A_A3_2', 'NSCLC_ALL_87_A_A4_1',
        'NSCLC_ALL_87_A_A4_2', 'NSCLC_ALL_87_A_A5_1',
        'NSCLC_ALL_87_A_A6_1'], dtype=object),
 array([ 499, 2080, 2131, 1882, 1742, 2487, 2984, 2398, 2095]))

In [19]:
df_to_load

,SLIDE,ROI,input file,save_group,roi_mappable,RoiID,Patient_ID,OS,DFS,Ev.O
0,NSCLC_ALL,88_B_B1_1,/project/Xie_Lab/zgu/xiao_multiplex/nsclc_tiff...,2,"88_B1,1","88_B1,1",88_422,898.0,586.0,1.0


In [10]:
rand_cytof_img.dict_feat

AttributeError: 'CytofImageTiff' object has no attribute 'dict_feat'

In [15]:
# store img into new dict
per_pt_roi_dict = {}
per_pt_roi_dict['temp'] = rand_cytof_img

per_pt_cohort = CytofCohort(cytof_images=per_pt_roi_dict, dir_out=None)
per_pt_cohort.batch_process_feature()
per_pt_cohort.generate_summary(accumul_type="ave")

Getting thresholds for cell ave of all markers.


['75normed_ave']

In [21]:
# df_feat = getattr(rand_cytof_img, 'df_feature_75normed')
# df_feat
df = rand_cytof_img.get_binary_pos_express_df(feature_name='75normed', accumul_type='ave')
df

,Myelope_276((2996))Y89-Y89_cell_ave,FSP1 S1_2263((3411))In113-In113_cell_ave,SMA_174((3277))In115-In115_cell_ave,117Sn-Sn117_cell_ave,Histone_126((2797))Pr141-Pr141_cell_ave,fap_323((3412))Nd142-Nd142_cell_ave,HLA-DR_1849((3362))Nd143-Nd143_cell_ave,CD146_22((3259))Nd144-Nd144_cell_ave,Cadheri_2088((2893))Nd145-Nd145_cell_ave,Carboni_2443((2757))Nd146-Nd146_cell_ave,...,CD4_2293((3000))Yb171-Yb171_cell_ave,CD31_1859((3370))Yb172-Yb172_cell_ave,anti-Hu_1950((2832))Yb173-Yb173_cell_ave,CCL21 6_2177((2889))Yb174-Yb174_cell_ave,panCyto_234((2745))Lu175-Lu175_cell_ave,K-Cadhe_2600((3417))Yb176-Yb176_cell_ave,Ki-67_142((3418))Pt194-Pt194_cell_ave,Caveoli_1945((2899))Pt195-Pt195_cell_ave,206Pb-Pb206_cell_ave,CD15_627((2997))Bi209-Bi209_cell_ave
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,True,True,False,False,False,True,False,False,...,False,True,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,True
4,False,False,True,False,False,False,False,False,True,False,...,False,True,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1070,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
1071,False,False,False,True,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1072,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1073,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False


In [23]:
panck_col = "panCyto_234((2745))Lu175-Lu175_cell_ave"
np.sum(df[panck_col])

195

In [25]:
len(df)

1075